# Lab Test Trends — Silver to Gold

Builds `careplus_db_ws.gold.lab_test_trends` with one row per test type per calendar month, including abnormal/critical rates and month-over-month abnormal rate changes via LAG().

In [0]:
%sql
CREATE OR REPLACE TABLE careplus_db_ws.gold.lab_test_trends
COMMENT 'Lab test trends — one row per test type per calendar month with abnormal rates and month-over-month changes'
AS
WITH monthly_agg AS (
  SELECT
    lr.test_name,
    YEAR(lr.test_date)                                                AS year,
    MONTH(lr.test_date)                                               AS month,
    COUNT(*)                                                          AS total_tests,
    COUNT(CASE WHEN lr.result_status = 'Abnormal' THEN 1 END)        AS abnormal_test_count,
    COUNT(CASE WHEN lr.result_status = 'Critical' THEN 1 END)        AS critical_test_count
  FROM careplus_db_ws.silver.lab_results lr
  GROUP BY lr.test_name, YEAR(lr.test_date), MONTH(lr.test_date)
),
with_rates AS (
  SELECT
    ma.*,
    CAST(ma.abnormal_test_count * 100.0 / ma.total_tests AS DECIMAL(5,2))  AS abnormal_rate_pct,
    CAST(ma.critical_test_count * 100.0 / ma.total_tests AS DECIMAL(5,2))  AS critical_rate_pct,
    LAG(CAST(ma.abnormal_test_count * 100.0 / ma.total_tests AS DECIMAL(5,2)))
      OVER (PARTITION BY ma.test_name ORDER BY ma.year, ma.month)          AS prev_month_abnormal_rate_pct
  FROM monthly_agg ma
)
SELECT
  wr.test_name,
  wr.year,
  wr.month,
  c.month_name,
  c.quarter_name,
  wr.total_tests,
  wr.abnormal_test_count,
  wr.critical_test_count,
  wr.abnormal_rate_pct,
  wr.critical_rate_pct,
  wr.prev_month_abnormal_rate_pct,
  CAST(
    CASE
      WHEN wr.prev_month_abnormal_rate_pct IS NULL THEN NULL
      ELSE wr.abnormal_rate_pct - wr.prev_month_abnormal_rate_pct
    END AS DECIMAL(6,2)
  ) AS abnormal_rate_change_pct
FROM with_rates wr
LEFT JOIN careplus_db_ws.silver.calender c
  ON c.date_key = CAST(date_format(make_date(wr.year, wr.month, 1), 'yyyyMMdd') AS INT)

In [0]:
%sql
SELECT
  COUNT(*)                    AS total_rows,
  COUNT(DISTINCT test_name)   AS distinct_tests,
  COUNT(DISTINCT year)        AS distinct_years,
  MIN(year)                   AS min_year,
  MAX(year)                   AS max_year,
  AVG(total_tests)            AS avg_monthly_tests
FROM careplus_db_ws.gold.lab_test_trends

In [0]:
%sql
SELECT * FROM careplus_db_ws.gold.lab_test_trends
ORDER BY test_name, year, month
LIMIT 10